In [1]:
import os
import pandas as pd
from datetime import datetime
import openai
import time

month_to_spanish = {
    1: "enero",
    2: "febrero",
    3: "marzo",
    4: "abril",
    5: "mayo",
    6: "junio",
    7: "julio",
    8: "agosto",
    9: "septiembre",
    10: "octubre",
    11: "noviembre",
    12: "diciembre"
}

def import_mananeras(base_path, year, month_number, day, all_man = False,
                     file_name = 'csv_por_participante/PRESIDENTE ANDRES MANUEL LOPEZ OBRADOR.csv' ):
    
    """
    Imports files from mananeras folder

    Args:
        base_path (str): The root path containing year/month-year/month day, year folders.
        year (int): year where you want to extract info
        month (int): month from where you want to extract info
        file_name (str): Name of the specific file to import. 
        all (bool): if True, imports the whole mananera

    Returns:
        list of pandas.DataFrame: List of DataFrames for the files imported.
    """
    
    month_to_spanish = {
    1: "enero",
    2: "febrero",
    3: "marzo",
    4: "abril",
    5: "mayo",
    6: "junio",
    7: "julio",
    8: "agosto",
    9: "septiembre",
    10: "octubre",
    11: "noviembre",
    12: "diciembre"}

    month_sp = month_to_spanish[month_number]
    if all_man == True: 
        file_name = f'{base_path}{year}/{month_number}-{year}/{month_sp} {day}, {year}/mananera_{day:02}_{month_number:02}_{year}.csv'
    else: 
        filename = file_name = f'{base_path}{year}/{month_number}-{year}/{month_sp} {day}, {year}/{file_name}'

    try:
        df = pd.read_csv(file_name)
    except: 
        print('No conference that day')
        df = pd.DataFrame()

    return df





/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
base_path = '../../data/02-conferences/raw/'

final = pd.DataFrame()
for y in range(2021, 2024): 
    int1 = pd.DataFrame()
    for mmm in range(1, 13): 
        int2 = pd.DataFrame()
        for ddd in range(1, 31): 
            df = import_mananeras(base_path, y, mmm, ddd, 
                      file_name='csv_por_participante/PREGUNTA.CSV')
            try:
                df = pd.DataFrame({'Texto': [" ".join(df['Texto'])]})
                df['month'] = mmm
                df['ddd'] = ddd
                df['y'] = y
            except:
                df = pd.DataFrame()
            

            int2 = pd.concat([int2, df])
        int1 = pd.concat([int1, int2])
    final = pd.concat([final, int1]).reset_index(drop=True)





No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conference that day
No conferen

In [50]:
import re

exclude_phrases = [
    "Hugo López Gatell", "Ciudad de México", "México", 
    "Presidente", "COVID", "Gobierno", 'Si', 'Y', 'También',
    'Gracias', 'Buenos', 'Cómo', 'Ayer', 'Se', 'Qué', 'Yo',
    'Preguntarle', 'Ahora', 'Una', 'QR', 'Cuál', 'Comisión', 
    'Federal'
]

# Function to filter sentences containing "días" and their adjacent ones
def filter_sentences(text):
    sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?)\s', text)
    result = []
    for i, sentence in enumerate(sentences):
        if 'días' in sentence:
            result.append(sentence)
            if i > 0:  # Add the previous sentence if it exists
                result.append(sentences[i-1])
            if i < len(sentences) - 1:  # Add the next sentence if it exists
                result.append(sentences[i+1])
    # Return only the relevant sentences joined together
    return ' '.join(set(result))

# Apply the function to extract relevant sentences
final['filtered_sentences'] = final['Texto'].apply(filter_sentences)

# Function to clean and filter the extracted sentences
def filter_text(text):
    # Remove excluded phrases
    for phrase in exclude_phrases:
        text = text.replace(phrase, "")
    # Remove extra spaces created during replacement
    text = re.sub(r'\s+', ' ', text).strip()
    # Filter words containing uppercase letters
    filtered_words = re.findall(r'\b\w*[A-Z]\w*\b', text)
    return ' '.join(filtered_words)

# Apply the function to the 'text' column
final['text_aux'] = final['Texto'].apply(filter_text)

In [ ]:
from openai import OpenAI
api_key = 'REDACTED_OPENAI_API_KEY'
openai.api_key = api_key

pred = []
for i in range(0, len(final)):
    texto = final['Texto'][i]
    try: 
        completion = openai.chat.completions.create(model="gpt-3.5-turbo",
                    messages=[
                        {"role": "system", "content": "Eres un asistente de investigación para un proyecto que busca analizar las conferencias de prensa de AMLO. Te dare el texto de todas las preguntas y conversación de los reporteros que fueron a la conferencia, regularmente estos se presentan, dicen su nombre y el medio al que pertenecen. Necesito que extraigas el nombre de todos los periodistas que lo mencionen, y el medio. Ponlos asi: Nombre, Medio | Nombre2, Medio2"},
                        {"role": "user", "content": texto}
                        ]
                )
        rev = completion.choices[0].message.content.strip()
        print(i)
    except: 
        rev = 'Not available'
        print('Not available')

    pred.append(rev)


final['periodistas'] = pred

final['date'] = (final['y'].astype(str) + '-' + 
                 final['month'].astype(str).str.zfill(2) + 
                 '-' + final['ddd'].astype(str).str.zfill(2))


# Step 2: Convert the string into a proper date variable
final['date'] = pd.to_datetime(final['date'], 
                               format='%Y-%m-%d')
final

final.to_parquet('../../data/02-conferences/auxiliar/periodistas_2021_2023.parquet')

In [92]:
# Step 1: Split the text column by commas
df_split = df_long['reporter'].str.split(',', expand=True)

# Step 2: Assign columns for reporter and outlets
df_long['reporter'] = df_split[0]  # The first column is the reporter
df_long['outlet1'] = df_split[1]  # The second column is the first outlet
df_long['outlet2'] = df_split[2]  # The third column is the second outlet (if exists)

# If you need more outlet columns, dynamically create them:
for i in range(3, df_split.shape[1]):
    df_long[f'outlet{i - 1}'] = df_split[i]

df_long

,date,reporter,outlet1,outlet2,outlet3,outlet4,outlet5,outlet6,outlet7,outlet8,...,outlet29,outlet30,outlet31,outlet32,outlet33,outlet34,outlet35,outlet36,outlet37,outlet38
0,2021-01-04,Héctor Tlatempa,Puntos Suspensivos Radio,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,2021-01-04,Meme Yamel,The México News,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,2021-01-04,Carlos Calzada,Radio Educación,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,2021-01-04,Demian Duarte,Sonora Power,Política,RockandRoll Radio,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,2021-01-04,Nuri Fernández,La Caracola,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2307,2023-12-21,Ángela Buitrago,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2308,2023-12-27,- José Sobrevilla,Noreste\n- Ernesto Ledesma,LordMolécula,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2309,2023-12-29,Beatriz Contreras,Gobierno de México,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2310,2023-12-29,Juan Hernández,Diario Basta,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
